In [ ]:
from google.colab import files
upload = files.upload()

Saving kaggle.json to kaggle.json


In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d rajansavaliya22/popular-movies-dataset

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [ ]:
import zipfile

with zipfile.ZipFile("popular-movies-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("movie_data")

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
movie = pd.read_csv('/content/movie_data/popular_movies.csv')
movie.head(1)

,title,budget,genres,original_language,overview,popularity,poster_path,release_date,revenue,runtime,spoken_languages,vote_average,vote_count,production_companies,production_countries
0,Final Destination Bloodlines,50000000,"[{'id': 27, 'name': 'Horror'}, {'id': 9648, 'n...",en,"Plagued by a violent recurring nightmare, coll...",1179.7021,/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,2025-05-14,274218717,110,"[{'english_name': 'English', 'iso_639_1': 'en'...",7.218,1035,"[{'id': 12, 'logo_path': '/2ycs64eqV5rqKYHyQK0...","[{'iso_3166_1': 'US', 'name': 'United States o..."


In [ ]:
credits = pd.read_csv('/content/movie_data/credits.csv')
credits.head(1)

,tmdb_id,title,cast,crew
0,574475,Final Destination Bloodlines,"[{'adult': False, 'gender': 1, 'id': 3480304, ...","[{'adult': False, 'gender': 2, 'id': 1181022, ..."


In [ ]:
movies = movie.merge(credits , on ='title' , how ='left')
movies.head(1)

,title,budget,genres,original_language,overview,popularity,poster_path,release_date,revenue,runtime,spoken_languages,vote_average,vote_count,production_companies,production_countries,tmdb_id,cast,crew
0,Final Destination Bloodlines,50000000,"[{'id': 27, 'name': 'Horror'}, {'id': 9648, 'n...",en,"Plagued by a violent recurring nightmare, coll...",1179.7021,/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,2025-05-14,274218717,110,"[{'english_name': 'English', 'iso_639_1': 'en'...",7.218,1035,"[{'id': 12, 'logo_path': '/2ycs64eqV5rqKYHyQK0...","[{'iso_3166_1': 'US', 'name': 'United States o...",574475.0,"[{'adult': False, 'gender': 1, 'id': 3480304, ...","[{'adult': False, 'gender': 2, 'id': 1181022, ..."


In [ ]:
movies.isna().sum()

,0
title,0
budget,0
genres,0
original_language,0
overview,103
popularity,0
poster_path,61
release_date,23
revenue,0
runtime,0


In [ ]:
movies = movies[movies['tmdb_id'].isna() == False]


In [ ]:
movies.dtypes

,0
title,object
budget,int64
genres,object
original_language,object
overview,object
popularity,float64
poster_path,object
release_date,object
revenue,int64
runtime,int64


In [ ]:
movies['tmdb_id'] = movies['tmdb_id'].astype('int')

In [ ]:
movies.dtypes

,0
title,object
budget,int64
genres,object
original_language,object
overview,object
popularity,float64
poster_path,object
release_date,object
revenue,int64
runtime,int64


In [ ]:
movies = movies[movies[['title','release_date']].duplicated() == False]

In [ ]:
movies[['title','tmdb_id']].duplicated().head(5)

,0
0,False
1,False
3,False
4,False
5,False


In [ ]:
movies.reset_index(drop = True,inplace = True)

In [ ]:
movies['release_date'] = pd.to_datetime(movies['release_date'])

In [ ]:
movies = movies[movies['poster_path'].isna() == False]

In [ ]:
movies.dtypes

,0
title,object
budget,int64
genres,object
original_language,object
overview,object
popularity,float64
poster_path,object
release_date,datetime64[ns]
revenue,int64
runtime,int64


In [ ]:

movies = movies[['tmdb_id','title','overview','genres','cast','crew','poster_path','production_companies']]

In [ ]:
import ast

In [ ]:
def convert(obj):
  gen_name = []
  for i in ast.literal_eval(obj):
        gen_name.append(i['name'])
  return gen_name

In [ ]:
movies['genres'] = movies['genres'].apply(convert)

In [ ]:
movies['genres']

,genres
0,"[Horror, Mystery]"
1,"[Family, Science Fiction, Comedy, Adventure]"
2,"[Thriller, Drama, Crime]"
3,"[Thriller, Action]"
4,"[Action, Family, Fantasy]"
...,...
9150,"[Drama, Action, Thriller, Crime]"
9151,[Action]
9152,[Documentary]
9153,"[Action, Adventure, Fantasy, Thriller]"


In [ ]:
def cast(text):
  names = []
  for i in ast.literal_eval(text):
    names.append(i['name'])
    if len(names) > 3 :
      break
  return names


In [ ]:
movies['cast'] = movies['cast'].apply(cast)

In [ ]:
movies['cast']

,cast
0,"[Kaitlyn Santa Juana, Teo Briones, Rya Kihlste..."
1,"[Maia Kealoha, Sydney Agudong, Chris Sanders, ..."
2,"[Taraji P. Henson, Sherri Shepherd, Teyana Tay..."
3,"[Rami Malek, Holt McCallany, Danny Sapani, Rac..."
4,"[Mason Thames, Nico Parker, Gerard Butler, Nic..."
...,...
9150,"[Tommy Lee Jones, Benicio del Toro, Connie Nie..."
9151,[]
9152,"[Johan Norberg, Per Ågren, Mattias Bengtsson, ..."
9153,"[Kento Yamazaki, Ryunosuke Kamiki, Yūsuke Isey..."


In [ ]:
def fetch_director(text):
    L = []
    for i in ast.literal_eval(text):
        if i['job'] == 'Director':
            L.append(i['name'])
    return L

In [ ]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [ ]:
movies.dtypes

,0
tmdb_id,int64
title,object
overview,object
genres,object
cast,object
crew,object
poster_path,object
production_companies,object


In [ ]:
movies['overview'] = movies['overview'].astype('str')

In [ ]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())

In [ ]:
movies['overview']

,overview
0,"[Plagued, by, a, violent, recurring, nightmare..."
1,"[The, wildly, funny, and, touching, story, of,..."
2,"[What, will, be, her, last, straw?, A, devasta..."
3,"[After, his, life, is, turned, upside, down, w..."
4,"[On, the, rugged, isle, of, Berk,, where, Viki..."
...,...
9150,"[In, the, wilderness, of, British, Columbia,, ..."
9151,"[Victoria,, an, intrepid, Latina, in, a, carte..."
9152,"[It's, been, suggested, that, Americans, would..."
9153,"[Morioh,, 1999—a, normally, quiet, and, peacef..."


In [ ]:
movies.head(1)

,tmdb_id,title,overview,genres,cast,crew,poster_path,production_companies
0,574475,Final Destination Bloodlines,"[Plagued, by, a, violent, recurring, nightmare...","[Horror, Mystery]","[Kaitlyn Santa Juana, Teo Briones, Rya Kihlste...","[Zach Lipovsky, Adam B. Stein]",/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,"[{'id': 12, 'logo_path': '/2ycs64eqV5rqKYHyQK0..."


In [ ]:
def company_name(text):
  company = []
  for i in ast.literal_eval(text):
    company.append(i['name'])
  return company


In [ ]:
movies['production_companies'] = movies['production_companies'].apply(company_name)

In [ ]:
def collapse(obj):
  list = []
  for i in obj :
    list.append(i.replace(' ',''))
  return list

In [ ]:
movies['genres'] = movies['genres'].apply(collapse)
movies['cast'] = movies['cast'].apply(collapse)
movies['crew'] = movies['crew'].apply(collapse)
movies['production_companies'] = movies['production_companies'].apply(collapse)

In [ ]:

movies.head(1)

,tmdb_id,title,overview,genres,cast,crew,poster_path,production_companies,teg,tags
0,574475,Final Destination Bloodlines,"[Plagued, by, a, violent, recurring, nightmare...","[Horror, Mystery]","[KaitlynSantaJuana, TeoBriones, RyaKihlstedt, ...","[ZachLipovsky, AdamB.Stein]",/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg,"[NewLineCinema, PracticalPictures, FreshmanYea...","[Horror, Mystery, Plagued, by, a, violent, rec...","[Horror, Mystery, Plagued, by, a, violent, rec..."


In [ ]:
movies['tags'] = movies['genres'] + movies['overview'] + movies['cast'] + movies['crew'] + movies['production_companies']

In [ ]:
new_df = movies[['tmdb_id','title','tags','poster_path']]

In [ ]:
new_df.reset_index(drop = True, inplace = True)

In [ ]:
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

In [ ]:
new_df.head(1)

,tmdb_id,title,tags,poster_path
0,574475,Final Destination Bloodlines,Horror Mystery Plagued by a violent recurring ...,/6WxhEvFsauuACfv8HyoVX6mZKFj.jpg


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=4000, stop_words='english')

In [ ]:
vector = cv.fit_transform(new_df['tags']).toarray()

In [ ]:
vector

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity(vector)
similarity

array([[1.        , 0.11128298, 0.        , ..., 0.        , 0.03012376,
        0.13764944],
       [0.11128298, 1.        , 0.        , ..., 0.        , 0.03184649,
        0.09701425],
       [0.        , 0.        , 1.        , ..., 0.        , 0.03282661,
        0.15      ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.01770536,
        0.        ],
       [0.03012376, 0.03184649, 0.03282661, ..., 0.01770536, 1.        ,
        0.05252257],
       [0.13764944, 0.09701425, 0.15      , ..., 0.        , 0.05252257,
        1.        ]])

In [ ]:
def recommend(movie):
  index = new_df[new_df['title'] == movie].index[0]
  distances = sorted(list(enumerate(similarity[index])), reverse=True, key = lambda x: x[1])
  for i in distances[1:6]:
        print(new_df.iloc[i[0]].title)

In [ ]:
recommend('Pulp Fiction')

The Beekeeper 2
Paradise City
Damaged
Go
Dead Man Down
